# EDA conjunto: conectores entre las cuatro fuentes

## 1. Objetivo y alcance

Pedí comprobar si puedo conectar ventas físicas, pedidos online, inventario y tipos de cambio, sin repetir los perfiles individuales. Este análisis **cruza las cuatro fuentes actuales**: evalúo cobertura referencial, cardinalidad, fechas, costo vigente y conservación de registros. No recalculo distribuciones aisladas ni presento respuestas finales de negocio.

Trabajo al nivel de registro de origen en cada canal. Uso el catálogo y las equivalencias explícitas de inventory como dimensiones; no uno ventas físicas con pedidos online como si fueran una misma transacción. Mantengo sin aplicar los candidatos por nombre y número. Una coincidencia técnica no confirma unidad física, estatus comercial ni ausencia de doble conteo.

## 2. Lectura y versión conjunta

Leo del Parquet únicamente claves y medidas comerciales; excluyo los campos personales incluso de la lectura. Conservo las columnas originales y añado derivadas en copias. `dia`, desde fecha_hora POS o fecha online, elimina la hora sin asignar zona; los textos no interpretables producirían NaT y detendrían el cruce. `dia` en snapshots y tasas interpreta fecha. Comparo las huellas con los informes individuales actuales, no con el estudio histórico de sales.

In [1]:
# Identifico las fuentes actuales y preparo copias sin datos personales para los cruces.
from pathlib import Path
import hashlib, json, re
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

raiz = Path.cwd() if (Path.cwd() / 'datos').is_dir() else Path.cwd().parent
assert (raiz / 'datos').is_dir(), 'Ejecutar desde la raíz o notebooks/'
tablas = []
def mostrar(titulo, tabla, leyenda):
    tabla = tabla.reset_index(drop=True)
    display(Markdown('**' + titulo + '**\n\n**Leyenda:** ' + leyenda + ' Muestro todas las filas; omito el índice de pandas.'))
    display(tabla)
    tablas.append((titulo, tabla.copy(), leyenda))

archivos = {'sales':'sales.csv', 'inventory':'inventory.json', 'ecommerce':'ecommerce_orders.parquet', 'exchange_rates':'exchange_rates.csv'}
huellas = {k: hashlib.sha256((raiz/'datos'/v).read_bytes()).hexdigest() for k,v in archivos.items()}
versiones = []
for fuente, archivo in archivos.items():
    informe = (raiz/'docs/02_exploracion'/f'resultados_{fuente}.md').read_text(encoding='utf-8')
    anteriores = re.findall(r'\b[a-f0-9]{64}\b', informe)
    versiones.append({'Fuente': archivo, 'SHA-256': huellas[fuente], 'Coincide con informe individual': huellas[fuente] in anteriores})
mostrar('Manifiesto de esta conciliación', pd.DataFrame(versiones), 'Identifico los cuatro archivos; SHA-256 es una huella de bytes, no una certificación comercial. Una diferencia exige documentar la nueva versión antes de interpretar resultados.')
assert all(v['Coincide con informe individual'] for v in versiones), 'Cambio de fuente: documentar diferencia de huellas'
sales = pd.read_csv(raiz/'datos/sales.csv')
online = pd.read_parquet(raiz/'datos/ecommerce_orders.parquet', columns=['order_id','fecha','product_handle','cantidad','amount','currency'])
inventario = json.loads((raiz/'datos/inventory.json').read_text(encoding='utf-8-sig'))
mapas = pd.DataFrame(inventario['sku_mappings'])
productos = pd.DataFrame(inventario['catalogo']['productos'])
tiendas = pd.DataFrame(inventario['tiendas_info'])
stock = pd.DataFrame(inventario['snapshots'])
tasas = pd.read_csv(raiz/'datos/exchange_rates.csv')
sales['dia'] = pd.to_datetime(sales['fecha_hora'], format='%d/%m/%Y %H:%M', errors='coerce').dt.normalize()
online['dia'] = pd.to_datetime(online['fecha'], format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.normalize()
stock['dia'] = pd.to_datetime(stock['fecha'], format='%Y-%m-%d', errors='coerce')
tasas['dia'] = pd.to_datetime(tasas['fecha'], format='%Y-%m-%d', errors='coerce')
assert all(d['dia'].notna().all() for d in [sales, online, stock, tasas])

**Manifiesto de esta conciliación**

**Leyenda:** Identifico los cuatro archivos; SHA-256 es una huella de bytes, no una certificación comercial. Una diferencia exige documentar la nueva versión antes de interpretar resultados. Muestro todas las filas; omito el índice de pandas.

,Fuente,SHA-256,Coincide con informe individual
0,sales.csv,1a456213c5590cf8c8d93438caf52c872ba4abd64a1b7003c74cc43acec8d01b,True
1,inventory.json,2355c6cab9ff00a707b5505a464a8f7e1a9bf5a2aa6c7a2b6b008f5299c8eada,True
2,ecommerce_orders.parquet,00c560b39425fe7f0088fbe20c66c2b8861d10c2c7006ba6e7335cdd6dfc856e,True
3,exchange_rates.csv,a0e827ec1e048cf397584eca3d692cb5b5b1f6b477fb78efca2fdbd2f488bddf,True


## 3. Contratos de unión: clave y granularidad

Compruebo la unicidad del lado derecho antes de cada unión muchos a uno (`m:1`): varias filas de movimientos pueden apuntar a una sola fila de dimensión. Excluyo claves nulas de las dimensiones de equivalencias para impedir que pandas enlace nulos entre sí. No elimino duplicados para hacer pasar una validación: una repetición detendría la ejecución.

`sku_erp` se incorpora desde sku_mappings usando sku POS o product_handle, con igualdad literal; no normalizo ni completo claves. `estado_enlace` diferencia ausencia de equivalencia, ERP nulo, ERP fuera de catálogo y enlace exacto. Esto permite separar falta de mapeo de falta de costo o snapshot. Las claves de las tablas de diagnóstico son conteos; no expongo identificadores de clientes.

In [2]:
# Verifico cardinalidad y cobertura sin rellenar equivalencias faltantes.
contratos = []
def contrato(nombre, dimension, claves):
    nulos = int(dimension[claves].isna().any(axis=1).sum())
    repetidas = int(dimension.duplicated(claves).sum())
    contratos.append({'Conector':nombre, 'Clave derecha': ', '.join(claves), 'Filas derechas':len(dimension), 'Claves nulas':nulos, 'Repeticiones adicionales':repetidas})
    assert nulos == 0 and repetidas == 0, nombre

mp = mapas.loc[mapas['sku_pos'].notna(), ['sku_pos','sku_erp']]
mh = mapas.loc[mapas['handle'].notna(), ['handle','sku_erp']]
contrato('POS a equivalencia', mp, ['sku_pos'])
contrato('Online a equivalencia', mh, ['handle'])
contrato('ERP a producto', productos, ['sku_erp'])
contrato('POS a tienda', tiendas, ['tienda_id'])
contrato('Movimiento a saldo diario', stock, ['dia','tienda_id','sku_erp'])
contrato('Moneda extranjera a tasa diaria', tasas, ['dia','currency'])
mostrar('Cardinalidad de los conectores', pd.DataFrame(contratos), 'Claves candidatas del lado derecho; conteos antes de unir. La dimensión de handles excluye sus nulos; no sustituyo duplicados por una fila arbitraria.')

def enlazar(origen, clave, dimension, clave_derecha):
    salida = origen.merge(dimension, left_on=clave, right_on=clave_derecha, how='left', validate='m:1', indicator='_enlace')
    salida['estado_enlace'] = np.select([salida['_enlace'].eq('left_only'), salida['sku_erp'].isna(), ~salida['sku_erp'].isin(productos['sku_erp'])], ['Sin equivalencia', 'Equivalencia con ERP nulo', 'ERP fuera de catálogo'], default='ERP exacto en catálogo')
    assert len(salida) == len(origen)
    return salida.drop(columns='_enlace')
fisico = enlazar(sales, 'sku', mp, 'sku_pos')
web = enlazar(online, 'product_handle', mh, 'handle')
cobertura = []
for canal, d, clave, moneda, importe in [('Físico',fisico,'sku','moneda','monto'), ('Online',web,'product_handle','currency','amount')]:
    for (estado, divisa), g in d.groupby(['estado_enlace',moneda], dropna=False):
        cobertura.append({'Canal':canal, 'Estado':estado, 'Moneda':divisa, 'Claves de origen':g[clave].nunique(), 'Registros':len(g), 'Cantidad registrada':g['cantidad'].sum(), 'Importe original':g[importe].sum(), 'Cobertura registros del canal %':100*len(g)/len(d)})
mostrar('Cobertura real de producto por canal', pd.DataFrame(cobertura), 'Toda la ventana de cada canal; importes en la moneda indicada, cantidades sin equivalencia física confirmada. Cada registro pertenece a un solo estado. Claves se cuentan dentro de moneda; no se suman como claves únicas globales.')
tiendas_pos = fisico.merge(tiendas[['tienda_id','timezone']], how='left', on='tienda_id', validate='m:1', indicator='_tienda')
erp_pos = set(fisico.loc[fisico.estado_enlace.eq('ERP exacto en catálogo'),'sku_erp'])
erp_web = set(web.loc[web.estado_enlace.eq('ERP exacto en catálogo'),'sku_erp'])
erp_stock = set(stock['sku_erp'])
mostrar('Puentes de producto y tienda', pd.DataFrame([
    {'Medida':'ERP presentes en ambos canales con enlace exacto', 'Resultado':len(erp_pos & erp_web)},
    {'Medida':'ERP físicos enlazados sin presencia global en snapshots', 'Resultado':len(erp_pos - erp_stock)},
    {'Medida':'ERP online enlazados sin presencia global en snapshots', 'Resultado':len(erp_web - erp_stock)},
    {'Medida':'Registros POS sin tienda en catálogo', 'Resultado':int(tiendas_pos['_tienda'].eq('left_only').sum())},
    {'Medida':'Tiendas de snapshots fuera del catálogo', 'Resultado':len(set(stock.tienda_id)-set(tiendas.tienda_id))}
]), 'Intersecciones calculadas en toda la cobertura. La presencia global de un ERP en snapshots no demuestra saldo disponible para una tienda, fecha o pedido concreto.')

**Cardinalidad de los conectores**

**Leyenda:** Claves candidatas del lado derecho; conteos antes de unir. La dimensión de handles excluye sus nulos; no sustituyo duplicados por una fila arbitraria. Muestro todas las filas; omito el índice de pandas.

,Conector,Clave derecha,Filas derechas,Claves nulas,Repeticiones adicionales
0,POS a equivalencia,sku_pos,65,0,0
1,Online a equivalencia,handle,27,0,0
2,ERP a producto,sku_erp,70,0,0
3,POS a tienda,tienda_id,40,0,0
4,Movimiento a saldo diario,"dia, tienda_id, sku_erp",230776,0,0
5,Moneda extranjera a tasa diaria,"dia, currency",730,0,0


**Cobertura real de producto por canal**

**Leyenda:** Toda la ventana de cada canal; importes en la moneda indicada, cantidades sin equivalencia física confirmada. Cada registro pertenece a un solo estado. Claves se cuentan dentro de moneda; no se suman como claves únicas globales. Muestro todas las filas; omito el índice de pandas.

,Canal,Estado,Moneda,Claves de origen,Registros,Cantidad registrada,Importe original,Cobertura registros del canal %
0,Físico,ERP exacto en catálogo,MXN,60,74141,114284,26989097.62,85.722049
1,Físico,Equivalencia con ERP nulo,MXN,5,6132,9552,2524147.68,7.089837
2,Físico,Sin equivalencia,MXN,5,6217,9547,1434008.52,7.188114
3,Online,ERP exacto en catálogo,EUR,23,335,444,7667.73,3.367850
4,Online,ERP exacto en catálogo,MXN,23,4875,6463,2200251.65,49.009752
5,Online,ERP exacto en catálogo,USD,23,1749,2382,46201.28,17.583191
6,Online,Equivalencia con ERP nulo,EUR,4,46,54,915.45,0.462451
7,Online,Equivalencia con ERP nulo,MXN,4,799,1052,318970.04,8.032573
8,Online,Equivalencia con ERP nulo,USD,4,294,403,7053.64,2.955665
9,Online,Sin equivalencia,EUR,6,89,115,1302.68,0.894742


**Puentes de producto y tienda**

**Leyenda:** Intersecciones calculadas en toda la cobertura. La presencia global de un ERP en snapshots no demuestra saldo disponible para una tienda, fecha o pedido concreto. Muestro todas las filas; omito el índice de pandas.

,Medida,Resultado
0,ERP presentes en ambos canales con enlace exacto,23
1,ERP físicos enlazados sin presencia global en snapshots,0
2,ERP online enlazados sin presencia global en snapshots,0
3,Registros POS sin tienda en catálogo,0
4,Tiendas de snapshots fuera del catálogo,0


**Interpretación e importancia.** Puedo usar inventory como puente parcial de identidad entre los canales y como dimensión de tienda del POS. Las cantidades e importes de filas sin enlace permanecen visibles: perderlas con un inner join sesgaría margen y rotación. Compartir un producto no identifica una transacción común. No vinculo pedidos a tienda desde dirección de envío ni uso IDs distintos como prueba de ausencia de doble conteo.

## 4. Compatibilidad temporal para los cruces

Calculo ventanas por intersección de extremos y presencia de días en las fuentes participantes. Es cobertura global diaria, no cobertura de cada producto, tienda o moneda. Una ventana común no prueba que los extractos estén completos. El corte propuesto depende de los datos y no de la fecha de ejecución; mantengo la hora original y no afirmo que el día POS equivalga al día contable de Shopify.

In [3]:
# Comparo calendarios únicamente para las conexiones que requieren fechas compatibles.
ventanas = []
for nombre, fuentes in [('POS con snapshots',[sales,stock]), ('Online con tasas',[online,tasas]), ('Cuatro fuentes',[sales,stock,online,tasas])]:
    inicio = max(d.dia.min() for d in fuentes)
    fin = min(d.dia.max() for d in fuentes)
    dias = pd.date_range(inicio, fin)
    comunes = set(dias)
    for d in fuentes:
        comunes &= set(d.dia)
    ventanas.append({'Cruce':nombre, 'Inicio común':str(inicio.date()), 'Fin común':str(fin.date()), 'Días entre extremos':len(dias), 'Días presentes en todas':len(comunes)})
mostrar('Ventanas utilizables de conexión', pd.DataFrame(ventanas), 'Días calendario con alguna fila por fuente; la cobertura por divisa y por tienda–SKU se valida más adelante. No relleno fechas ausentes.')
inicio_stock, fin_stock = stock.dia.min(), stock.dia.max()
calendario_canales = []
for mes in pd.period_range(min(sales.dia.min(),online.dia.min()), max(sales.dia.max(),online.dia.max()), freq='M'):
    dias_mes = set(pd.date_range(mes.start_time.normalize(),mes.end_time.normalize()))
    calendario_canales.append({'Mes':str(mes), 'Días POS':len(set(sales.dia)&dias_mes), 'Días online':len(set(online.dia)&dias_mes), 'Días snapshots':len(set(stock.dia)&dias_mes), 'Días FX':len(set(tasas.dia)&dias_mes)})
mostrar('Calendario comparado para las ventanas del reto', pd.DataFrame(calendario_canales), 'Días con alguna fila por fuente en cada mes observado; cero significa ausencia de registros en esa fuente, no ventas ni stock cero. Este cuadro compara cobertura, no repite importes mensuales individuales.')

**Ventanas utilizables de conexión**

**Leyenda:** Días calendario con alguna fila por fuente; la cobertura por divisa y por tienda–SKU se valida más adelante. No relleno fechas ausentes. Muestro todas las filas; omito el índice de pandas.

,Cruce,Inicio común,Fin común,Días entre extremos,Días presentes en todas
0,POS con snapshots,2025-10-01,2026-03-31,182,182
1,Online con tasas,2025-04-01,2026-03-31,365,365
2,Cuatro fuentes,2025-10-01,2026-03-31,182,182


**Calendario comparado para las ventanas del reto**

**Leyenda:** Días con alguna fila por fuente en cada mes observado; cero significa ausencia de registros en esa fuente, no ventas ni stock cero. Este cuadro compara cobertura, no repite importes mensuales individuales. Muestro todas las filas; omito el índice de pandas.

,Mes,Días POS,Días online,Días snapshots,Días FX
0,2024-10,31,0,0,0
1,2024-11,30,0,0,0
2,2024-12,31,0,0,0
3,2025-01,31,0,0,0
4,2025-02,28,0,0,0
5,2025-03,31,0,0,0
6,2025-04,30,30,0,30
7,2025-05,31,31,0,31
8,2025-06,30,30,0,30
9,2025-07,31,31,0,31


**Interpretación e importancia.** Puedo comparar actividad por canal dentro del año común POS–online, pero el primer mes online no tiene una base online del mes anterior. Para rotación restrinjo las ventas al período con inventario y después compruebo tienda–SKU–día; la intersección temporal por sí sola no basta. FX es necesario para moneda extranjera: su ausencia antes del período online no impide conservar POS en MXN.

## 5. Ventas físicas frente a snapshots: cobertura utilizable

`stock_num`, desde cantidad_en_stock, convierte textos no interpretables a NaN sin convertirlos a cero. Uno por **dia, tienda_id y sku_erp**, con validación m:1; unir solo por producto multiplicaría filas y mezclaría tiendas y fechas. `estado_stock` prioriza falta de ERP, fecha fuera del período de snapshots, ausencia de fila y después saldo desconocido, cero, positivo u otro valor numérico. Un snapshot ausente no es un nulo de saldo ni una existencia cero.

Conservo todos los comprobantes en el diagnóstico global y separo I en el semestre como escenario de elegibilidad, sin confirmar que sea la definición final de ventas. Los porcentajes usan el total de cada población, incluidos los registros sin mapeo. No sumo saldos diarios como inventario disponible ni igualo ventas con variaciones de stock: faltan compras, traslados y ajustes.

In [4]:
# Distingo ausencia de enlace, falta de snapshot y saldo desconocido en cada movimiento físico.
stock['stock_num'] = pd.to_numeric(stock['cantidad_en_stock'], errors='coerce')
fisico = fisico.merge(stock[['dia','tienda_id','sku_erp','stock_num']], on=['dia','tienda_id','sku_erp'], how='left', validate='m:1', indicator='_snapshot')
fisico['estado_stock'] = np.select([
    ~fisico.estado_enlace.eq('ERP exacto en catálogo'),
    ~fisico.dia.between(inicio_stock,fin_stock),
    fisico['_snapshot'].eq('left_only'), fisico.stock_num.isna(),
    fisico.stock_num.eq(0), fisico.stock_num.gt(0)
], ['Sin ERP conciliado','Fuera de ventana de snapshots','Sin fila de snapshot','Saldo desconocido','Saldo cero','Saldo positivo'], default='Otro saldo numérico')
resumen_stock = []
escenarios = [('Todos los comprobantes; toda la cobertura', fisico), ('Solo I; ventana de snapshots',fisico.loc[fisico.tipo_comprobante.eq('I') & fisico.dia.between(inicio_stock,fin_stock)])]
for poblacion, d in escenarios:
    assert d.moneda.eq('MXN').all()
    for estado, g in d.groupby('estado_stock'):
        resumen_stock.append({'Población':poblacion,'Estado de conexión':estado,'Registros':len(g),'Cantidad registrada':g.cantidad.sum(),'Importe original MXN':g.monto.sum(),'Registros de la población %':100*len(g)/len(d)})
mostrar('Efecto de la conexión al inventario', pd.DataFrame(resumen_stock), 'Estados excluyentes por movimiento; importes MXN sin signo analítico aplicado y cantidades originales. Solo I es un escenario, no ingreso neto confirmado.')
mensual_stock = (fisico.loc[fisico.tipo_comprobante.eq('I') & fisico.dia.between(inicio_stock,fin_stock)]
    .assign(Mes=lambda d:d.dia.dt.strftime('%Y-%m')).groupby(['Mes','estado_stock']).size().unstack(fill_value=0).reset_index().rename_axis(columns=None))
mostrar('Cobertura de conexión por mes en el escenario I', mensual_stock, 'Conteos de registros I durante la ventana de snapshots, por estado. Cero en una celda significa que ningún movimiento cae en ese estado; no imputa stock.')

**Efecto de la conexión al inventario**

**Leyenda:** Estados excluyentes por movimiento; importes MXN sin signo analítico aplicado y cantidades originales. Solo I es un escenario, no ingreso neto confirmado. Muestro todas las filas; omito el índice de pandas.

,Población,Estado de conexión,Registros,Cantidad registrada,Importe original MXN,Registros de la población %
0,Todos los comprobantes; toda la cobertura,Fuera de ventana de snapshots,49585,76258,18012284.63,57.330327
1,Todos los comprobantes; toda la cobertura,Saldo cero,593,914,222341.22,0.685628
2,Todos los comprobantes; toda la cobertura,Saldo desconocido,217,327,72619.11,0.250896
3,Todos los comprobantes; toda la cobertura,Saldo positivo,10473,16240,3810220.24,12.108914
4,Todos los comprobantes; toda la cobertura,Sin ERP conciliado,12349,19099,3958156.20,14.277951
5,Todos los comprobantes; toda la cobertura,Sin fila de snapshot,13273,20545,4871632.42,15.346283
6,Solo I; ventana de snapshots,Saldo cero,565,866,210818.66,2.070507
7,Solo I; ventana de snapshots,Saldo desconocido,213,317,70228.27,0.780563
8,Solo I; ventana de snapshots,Saldo positivo,9950,15437,3599073.91,36.462914
9,Solo I; ventana de snapshots,Sin ERP conciliado,3873,5988,1245482.00,14.193052


**Cobertura de conexión por mes en el escenario I**

**Leyenda:** Conteos de registros I durante la ventana de snapshots, por estado. Cero en una celda significa que ningún movimiento cae en ese estado; no imputa stock. Muestro todas las filas; omito el índice de pandas.

,Mes,Saldo cero,Saldo desconocido,Saldo positivo,Sin ERP conciliado,Sin fila de snapshot
0,2025-10,86,42,1729,642,2187
1,2025-11,97,34,1658,627,2092
2,2025-12,97,30,1709,703,2139
3,2026-01,103,45,1667,644,2170
4,2026-02,92,27,1509,617,1971
5,2026-03,90,35,1678,640,2128


**Interpretación e importancia.** La identidad ERP puede ser correcta y aun así faltar el par tienda–producto en inventario. Por ello, para rotación necesito una población compatible en numerador y denominador y cobertura de saldos conocidos; no extrapolo el subconjunto conciliado a todo el canal. Ventas con saldo cero el mismo día son una coincidencia temporal observada: sin hora ni definición del snapshot no prueban sobreventa ni quiebre continuo.

## 6. Costo vigente: conexión temporal al catálogo

Expando cost_history en `vigencia` (fecha_vigencia interpretada) y `costo_num` (costo_mxn numérico); detengo el análisis si encuentro fechas desconocidas, claves repetidas o costos no positivos. Con `merge_asof` busco por ERP la última vigencia menor o igual al día del registro, aceptando coincidencia en el mismo día. `dias_desde_vigencia` es día del movimiento menos vigencia; mide antigüedad, no vencimiento. No uso un costo futuro ni el costo actual para todo el histórico.

`estado_costo` distingue sin ERP, sin vigencia previa y costo temporal encontrado. Aplico el costo unitario global en MXN desde su vigencia hasta el siguiente cambio; no confirmo base fiscal, unidad física ni método contable. No calculo margen hasta que esas bases sean comparables.

In [5]:
# Vinculo cada movimiento a una sola vigencia previa del ERP y conservo los no conciliados.
costos = pd.DataFrame([{'sku_erp':p['sku_erp'], **c} for p in inventario['catalogo']['productos'] for c in p['cost_history']])
costos['vigencia'] = pd.to_datetime(costos['fecha_vigencia'], format='%Y-%m-%d', errors='coerce')
costos['costo_num'] = pd.to_numeric(costos['costo_mxn'],errors='coerce')
contrato('Movimiento a historial de costo', costos, ['sku_erp','vigencia'])
assert costos.vigencia.notna().all() and np.isfinite(costos.costo_num).all() and costos.costo_num.gt(0).all()
mostrar('Clave temporal del historial usado en el cruce', pd.DataFrame(contratos[-1:]), 'Cada ERP puede tener varias vigencias; la combinación ERP–vigencia debe ser única, con fecha conocida y costo finito positivo. La selección temporal devuelve a lo sumo una vigencia por movimiento.')
def agregar_costo(d):
    copia = d.copy()
    copia['_orden'] = range(len(copia))
    conocidos = copia.loc[copia.estado_enlace.eq('ERP exacto en catálogo')].sort_values('dia')
    unidos = pd.merge_asof(conocidos, costos[['sku_erp','vigencia','costo_num']].sort_values('vigencia'), left_on='dia',right_on='vigencia',by='sku_erp',direction='backward',allow_exact_matches=True)
    salida = copia.merge(unidos[['_orden','vigencia','costo_num']],on='_orden',how='left',validate='1:1').drop(columns='_orden')
    salida['estado_costo'] = np.select([~salida.estado_enlace.eq('ERP exacto en catálogo'),salida.vigencia.isna()], ['Sin ERP conciliado','Sin vigencia previa'],default='Costo temporal encontrado')
    salida['dias_desde_vigencia'] = (salida.dia-salida.vigencia).dt.days
    assert len(salida)==len(d) and salida.dias_desde_vigencia.dropna().ge(0).all()
    return salida
fisico = agregar_costo(fisico)
web = agregar_costo(web)
resumen_costos = []
for canal,d,moneda in [('Físico',fisico,'moneda'),('Online',web,'currency')]:
    for (estado,divisa),g in d.groupby(['estado_costo',moneda]):
        resumen_costos.append({'Canal':canal,'Moneda original':divisa,'Estado de costo':estado,'Registros':len(g),'ERP distintos':g.sku_erp.nunique(),'Días mínimos desde vigencia':g.dias_desde_vigencia.min(),'Días máximos desde vigencia':g.dias_desde_vigencia.max()})
mostrar('Cobertura de costo aplicable por fecha', pd.DataFrame(resumen_costos), 'Toda la ventana y comprobantes; cuento registros, no márgenes. Costos en MXN por unidad ERP pendiente de confirmar; los extremos son días calendario y quedan ausentes si no hay enlace.')

**Clave temporal del historial usado en el cruce**

**Leyenda:** Cada ERP puede tener varias vigencias; la combinación ERP–vigencia debe ser única, con fecha conocida y costo finito positivo. La selección temporal devuelve a lo sumo una vigencia por movimiento. Muestro todas las filas; omito el índice de pandas.

,Conector,Clave derecha,Filas derechas,Claves nulas,Repeticiones adicionales
0,Movimiento a historial de costo,"sku_erp, vigencia",282,0,0


**Cobertura de costo aplicable por fecha**

**Leyenda:** Toda la ventana y comprobantes; cuento registros, no márgenes. Costos en MXN por unidad ERP pendiente de confirmar; los extremos son días calendario y quedan ausentes si no hay enlace. Muestro todas las filas; omito el índice de pandas.

,Canal,Moneda original,Estado de costo,Registros,ERP distintos,Días mínimos desde vigencia,Días máximos desde vigencia
0,Físico,MXN,Costo temporal encontrado,74141,60,0.0,497.0
1,Físico,MXN,Sin ERP conciliado,12349,0,NaN,NaN
2,Online,EUR,Costo temporal encontrado,335,23,1.0,422.0
3,Online,MXN,Costo temporal encontrado,4875,23,0.0,429.0
4,Online,USD,Costo temporal encontrado,1749,23,0.0,422.0
5,Online,EUR,Sin ERP conciliado,135,0,NaN,NaN
6,Online,MXN,Sin ERP conciliado,2067,0,NaN,NaN
7,Online,USD,Sin ERP conciliado,786,0,NaN,NaN


**Interpretación e importancia.** La cobertura temporal del costo debe evaluarse después del enlace de producto. Una fila sin ERP no es un costo cero ni un producto sin costo en catálogo. El resultado habilita una comparación condicionada, pero no certifica márgenes: cantidad del canal, unidad ERP, impuestos, descuentos y signos requieren bases comparables.

## 7. Monedas: conector diario e independencia del mapeo

Uno online a tasas por dia y currency con m:1; `tasa_aplicada` toma rate_to_mxn para divisas y 1 para MXN como identidad matemática, sin exigir una fila MXN en FX. `estado_fx` distingue identidad MXN, tasa diaria positiva y ausencia o invalidez. Para POS en MXN no necesito cotización extranjera.

Calculo solo un **importe convertido diagnóstico**, amount × tasa, con Decimal construido desde la representación textual; no recupera una precisión original perdida en double. Comparo redondear cada registro a dos decimales frente a redondear su suma, usando ROUND_HALF_UP como escenario técnico. No establezco una política contable ni presento el convertido como ingreso reconocido. Mantengo monedas originales separadas y no aplico relleno temporal de tasas.

In [6]:
# Verifico cobertura FX por registro y cuantifico la sensibilidad a la etapa del redondeo.
assert np.isfinite(tasas.rate_to_mxn).all() and tasas.rate_to_mxn.gt(0).all()
web = web.merge(tasas[['dia','currency','rate_to_mxn']],on=['dia','currency'],how='left',validate='m:1')
web['tasa_aplicada'] = web.rate_to_mxn.where(web.currency.ne('MXN'),1.0)
web['estado_fx'] = np.select([web.currency.eq('MXN'),web.tasa_aplicada.notna() & web.tasa_aplicada.gt(0)], ['Identidad MXN','Tasa diaria positiva'], default='Sin tasa válida')
fx_resultados = []
for (divisa,estado),g in web.groupby(['currency','estado_fx']):
    convertidos = [Decimal(str(a))*Decimal(str(t)) for a,t in zip(g.amount,g.tasa_aplicada) if pd.notna(t) and t>0]
    por_linea = sum((v.quantize(Decimal('0.01'),rounding=ROUND_HALF_UP) for v in convertidos),Decimal(0))
    al_final = sum(convertidos,Decimal(0)).quantize(Decimal('0.01'),rounding=ROUND_HALF_UP)
    fx_resultados.append({'Moneda original':divisa,'Estado FX':estado,'Registros':len(g),'Registros convertibles':len(convertidos),'Suma MXN redondeada por registro':str(por_linea) if convertidos else 'No disponible','Suma MXN redondeada al final':str(al_final) if convertidos else 'No disponible','Diferencia MXN':str(por_linea-al_final) if convertidos else 'No disponible'})
mostrar('Conexión financiera y sensibilidad del redondeo',pd.DataFrame(fx_resultados),'Todo online, incluso productos no conciliados. Sumas en MXN bajo dos escenarios de redondeo a centavos; no son ingresos netos ni sustituyen importes originales.')
cadena = web.groupby(['estado_enlace','estado_costo','estado_fx','currency'],dropna=False).size().reset_index(name='Registros').rename(columns={'estado_enlace':'Producto','estado_costo':'Costo','estado_fx':'Conversión','currency':'Moneda original'})
mostrar('Cobertura conjunta online: producto, costo y moneda',cadena,'Conteo de registros en todas las combinaciones observadas. Una tasa disponible no repara un ERP faltante; un costo encontrado no identifica almacén ni valida unidad.')

**Conexión financiera y sensibilidad del redondeo**

**Leyenda:** Todo online, incluso productos no conciliados. Sumas en MXN bajo dos escenarios de redondeo a centavos; no son ingresos netos ni sustituyen importes originales. Muestro todas las filas; omito el índice de pandas.

,Moneda original,Estado FX,Registros,Registros convertibles,Suma MXN redondeada por registro,Suma MXN redondeada al final,Diferencia MXN
0,EUR,Tasa diaria positiva,470,470,214922.15,214922.16,-0.01
1,MXN,Identidad MXN,6942,6942,2912430.74,2912430.74,0.00
2,USD,Tasa diaria positiva,2535,2535,1101529.65,1101529.81,-0.16


**Cobertura conjunta online: producto, costo y moneda**

**Leyenda:** Conteo de registros en todas las combinaciones observadas. Una tasa disponible no repara un ERP faltante; un costo encontrado no identifica almacén ni valida unidad. Muestro todas las filas; omito el índice de pandas.

,Producto,Costo,Conversión,Moneda original,Registros
0,ERP exacto en catálogo,Costo temporal encontrado,Identidad MXN,MXN,4875
1,ERP exacto en catálogo,Costo temporal encontrado,Tasa diaria positiva,EUR,335
2,ERP exacto en catálogo,Costo temporal encontrado,Tasa diaria positiva,USD,1749
3,Equivalencia con ERP nulo,Sin ERP conciliado,Identidad MXN,MXN,799
4,Equivalencia con ERP nulo,Sin ERP conciliado,Tasa diaria positiva,EUR,46
5,Equivalencia con ERP nulo,Sin ERP conciliado,Tasa diaria positiva,USD,294
6,Sin equivalencia,Sin ERP conciliado,Identidad MXN,MXN,1268
7,Sin equivalencia,Sin ERP conciliado,Tasa diaria positiva,EUR,89
8,Sin equivalencia,Sin ERP conciliado,Tasa diaria positiva,USD,492


**Interpretación e importancia.** El conector FX y el conector de producto resuelven problemas independientes: puedo convertir un registro extranjero aunque no llegue a ERP, pero no asignarle costo o inventario automáticamente. La diferencia entre escenarios de redondeo justifica declarar cuándo se redondea antes de conciliar totales. No sumo canales como ingreso consolidado mientras no pueda controlar estatus y doble conteo.

## 8. Conservación y comprobaciones de los cruces

Compruebo que los left joins conserven cada registro, cantidades y sumas en su moneda; las sumas son controles técnicos, no cierres contables. Verifico de nuevo las huellas y que las columnas originales permanezcan iguales. La tolerancia 1e-8 solo absorbe diferencias de coma flotante al agregar. La asignación temporal no debe tener vigencias futuras y ningún enlace técnico puede multiplicar filas.

In [7]:
# Compruebo conservación de originales y que las fuentes permanezcan intactas tras conciliar.
controles = []
for canal,original,salida,id_col,moneda,importe in [('Físico',sales,fisico,'venta_id','moneda','monto'),('Online',online,web,'order_id','currency','amount')]:
    pd.testing.assert_frame_equal(original.reset_index(drop=True),salida[original.columns].reset_index(drop=True), check_dtype=True)
    assert salida[id_col].is_unique and len(salida)==len(original)
    for campo in ['cantidad',importe]:
        a=original.groupby(moneda)[campo].sum().sort_index()
        b=salida.groupby(moneda)[campo].sum().sort_index()
        assert a.index.equals(b.index) and np.allclose(a,b,rtol=0,atol=1e-8)
    controles.append({'Control':canal+': registros, originales y sumas conservados','Resultado':'Cumple'})
    validos = salida.loc[salida.vigencia.notna()]
    for erp,g in validos.groupby('sku_erp'):
        fechas = np.sort(costos.loc[costos.sku_erp.eq(erp),'vigencia'].to_numpy())
        esperado = fechas[np.searchsorted(fechas,g.dia.to_numpy(),side='right')-1]
        assert np.array_equal(esperado,g.vigencia.to_numpy())
    controles.append({'Control':canal+': última vigencia no futura contrastada con búsqueda ordenada','Resultado':'Cumple'})
for fuente,archivo in archivos.items():
    assert huellas[fuente] == hashlib.sha256((raiz/'datos'/archivo).read_bytes()).hexdigest()
controles.append({'Control':'Cuatro SHA-256 finales iguales a los iniciales','Resultado':'Cumple'})
mostrar('Controles finales de integración',pd.DataFrame(controles),'Validaciones ejecutables de este notebook. Ante una falla detengo la ejecución; no reparo fuentes ni oculto movimientos sin correspondencia.')

**Controles finales de integración**

**Leyenda:** Validaciones ejecutables de este notebook. Ante una falla detengo la ejecución; no reparo fuentes ni oculto movimientos sin correspondencia. Muestro todas las filas; omito el índice de pandas.

,Control,Resultado
0,"Físico: registros, originales y sumas conservados",Cumple
1,Físico: última vigencia no futura contrastada con búsqueda ordenada,Cumple
2,"Online: registros, originales y sumas conservados",Cumple
3,Online: última vigencia no futura contrastada con búsqueda ordenada,Cumple
4,Cuatro SHA-256 finales iguales a los iniciales,Cumple


### Interpretación actual del costo
La respuesta que compartí establece que costo_mxn es unitario en MXN y global para todas las tiendas, desde fecha_vigencia hasta el siguiente cambio. Aplico la última vigencia no futura; mantengo como limitaciones la unidad física y la base fiscal comparable. Los resultados guardados corresponden a la versión de fuentes identificada en este análisis.

## 9. Qué conexiones puedo usar

Puedo usar el SKU POS y el handle como entradas separadas a las equivalencias explícitas del ERP, tienda_id para enriquecer el POS y la clave compuesta día–tienda–ERP para consultar snapshots. Puedo conectar ERP con su última vigencia de costo no futura y día–moneda con la tasa diaria extranjera. La cobertura es parcial y queda cuantificada en las tablas; aplico uniones que conservan los movimientos no conciliados.

Para crecimiento por canal dispongo de un calendario común, condicionado por estados comerciales, signos y moneda. Para rotación necesito restringir las ventas y los saldos a la misma población y ventana; no incorporo online al inventario de tiendas sin almacén de cumplimiento. Para quiebres, las ventas coincidentes con cero no demuestran falta continua de disponibilidad. Para margen, el enlace de costo no confirma que cantidad, impuestos y base financiera sean comparables.

Mi análisis no establece una relación transaccional POS–online: no hay una referencia común que permita comprobar duplicación entre canales. No uso nombre, sufijo, cercanía horaria ni dirección de envío para inventarla. Mantengo esas limitaciones al construir el modelo.

## Documentación relacionada

- [Informe de resultados del análisis conjunto](../docs/02_exploracion/resultados_conjunto.md).

## Conclusión

Tengo comprobados los conectores técnicos y su cobertura entre las cuatro fuentes actuales, con protección contra multiplicación de filas y pérdida silenciosa de movimientos. Puedo avanzar al modelo con equivalencias exactas y excepciones visibles, conservando como límites la identidad incompleta, la cobertura tienda–producto, las unidades y la interpretación financiera y temporal.